<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/K%C3%A9pklasszifik%C3%A1ci%C3%B3s%20M%C3%B3dszerek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Képklasszifikációs Módszerek

Ebben a notebookban az alapvető **képklasszifikációs** módszereket vizsgáljuk.

## Tartalomjegyzék

1. Képklasszifikáció alapok
2. CNN architektúrák
3. Data augmentation
4. Transfer learning
5. Teljes pipeline

## 1. Képklasszifikáció alapok

### Feladat

| Input | Output |
|-------|--------|
| Kép $(H, W, C)$ | Osztály címke |
| RGB: $(224, 224, 3)$ | Softmax valószínűségek |

### Pipeline

```
Kép → Előfeldolgozás → CNN → Feature vector → FC → Softmax → Osztály
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T

np.random.seed(42)
torch.manual_seed(42)

# Szintetikus adathalmaz generálása
def generate_classification_data(n_samples=1000, img_size=32):
    """4 osztályú képklasszifikációs adat."""
    images = []
    labels = []

    for i in range(n_samples):
        img = np.zeros((3, img_size, img_size))
        label = i % 4

        # Alapszín és zaj
        noise = np.random.randn(3, img_size, img_size) * 0.1

        if label == 0:  # Piros kör
            cx, cy = img_size//2 + np.random.randint(-4, 5), img_size//2 + np.random.randint(-4, 5)
            r = 8 + np.random.randint(-2, 3)
            for x in range(img_size):
                for y in range(img_size):
                    if (x-cx)**2 + (y-cy)**2 < r**2:
                        img[0, x, y] = 1  # Red channel

        elif label == 1:  # Zöld négyzet
            sx = img_size//2 - 6 + np.random.randint(-3, 4)
            sy = img_size//2 - 6 + np.random.randint(-3, 4)
            size = 12 + np.random.randint(-2, 3)
            img[1, sx:sx+size, sy:sy+size] = 1  # Green channel

        elif label == 2:  # Kék háromszög
            cx, cy = img_size//2, img_size//2
            for x in range(img_size):
                for y in range(img_size):
                    if y > cy - 8 and y < cy + 8 and x > cx - (y - cy + 8) and x < cx + (y - cy + 8):
                        img[2, x, y] = 1  # Blue channel

        else:  # Sárga kereszt
            cx, cy = img_size//2, img_size//2
            img[0, cx-8:cx+8, cy-2:cy+2] = 1  # Red
            img[1, cx-8:cx+8, cy-2:cy+2] = 1  # Green (sárga = R+G)
            img[0, cx-2:cx+2, cy-8:cy+8] = 1
            img[1, cx-2:cx+2, cy-8:cy+8] = 1

        img = img + noise
        img = np.clip(img, 0, 1)

        images.append(img)
        labels.append(label)

    return np.array(images, dtype=np.float32), np.array(labels)

# Adat generálás
X, y = generate_classification_data(2000)
X_train, y_train = X[:1600], y[:1600]
X_test, y_test = X[1600:], y[1600:]

# Vizualizáció
class_names = ['Piros kör', 'Zöld négyzet', 'Kék háromszög', 'Sárga kereszt']

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, label in enumerate([0, 1, 2, 3]):
    idx = np.where(y == label)[0]
    for j in range(2):
        img = X[idx[j]].transpose(1, 2, 0)  # CHW -> HWC
        axes[j, i].imshow(img)
        axes[j, i].axis('off')
        if j == 0:
            axes[j, i].set_title(class_names[label])

plt.suptitle('Szintetikus képklasszifikációs adat', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. CNN architektúrák

### Egyszerű CNN

```
Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → FC → Softmax
```

In [ ]:
# Egyszerű CNN

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8 -> 4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Modern CNN BatchNorm-mal
class ModernCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Összehasonlítás
simple_cnn = SimpleCNN()
modern_cnn = ModernCNN()

print("SimpleCNN:")
print(f"  Paraméterek: {sum(p.numel() for p in simple_cnn.parameters()):,}")

print("\nModernCNN:")
print(f"  Paraméterek: {sum(p.numel() for p in modern_cnn.parameters()):,}")

In [ ]:
# Training loop

def train_model(model, train_loader, test_loader, epochs=30, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_correct += predicted.eq(y_batch).sum().item()
            train_total += y_batch.size(0)

        # Evaluation
        model.eval()
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                _, predicted = outputs.max(1)
                test_correct += predicted.eq(y_batch).sum().item()
                test_total += y_batch.size(0)

        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(train_correct / train_total)
        history['test_acc'].append(test_correct / test_total)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Loss={history['train_loss'][-1]:.4f}, "
                  f"Train Acc={history['train_acc'][-1]:.1%}, "
                  f"Test Acc={history['test_acc'][-1]:.1%}")

    return history

# Data loaders
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32)

# Training
print("SimpleCNN tanítás:")
simple_cnn = SimpleCNN()
history_simple = train_model(simple_cnn, train_loader, test_loader, epochs=30)

In [ ]:
print("\nModernCNN tanítás:")
modern_cnn = ModernCNN()
history_modern = train_model(modern_cnn, train_loader, test_loader, epochs=30)

# Összehasonlítás
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_simple['train_loss'], label='SimpleCNN')
axes[0].plot(history_modern['train_loss'], label='ModernCNN')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_simple['test_acc'], label='SimpleCNN')
axes[1].plot(history_modern['test_acc'], label='ModernCNN')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Data augmentation

### Gyakori augmentációk

| Típus | Leírás |
|-------|--------|
| Flip | Vízszintes/függőleges tükrözés |
| Rotation | Forgatás |
| Crop | Kivágás, méretezés |
| Color | Színek módosítása |
| Cutout | Random maszkolás |

In [ ]:
# Augmentációk vizualizálása

# Egy teszt kép
sample_img = X_train[0]  # (3, 32, 32)
sample_tensor = torch.FloatTensor(sample_img)

# Augmentációk
augmentations = {
    'Original': T.Compose([]),
    'HFlip': T.RandomHorizontalFlip(p=1.0),
    'VFlip': T.RandomVerticalFlip(p=1.0),
    'Rotate': T.RandomRotation(30),
    'ColorJitter': T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    'Affine': T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for ax, (name, aug) in zip(axes, augmentations.items()):
    torch.manual_seed(42)
    augmented = aug(sample_tensor)
    img = augmented.numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')

plt.suptitle('Data Augmentation példák', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Augmentált dataset

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform:
            x = self.transform(x)
        return x, self.y[idx]

# Training transforms
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
])

# Datasets
train_dataset_aug = AugmentedDataset(X_train, y_train, transform=train_transform)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)

# Training with augmentation
print("ModernCNN augmentációval:")
modern_cnn_aug = ModernCNN()
history_aug = train_model(modern_cnn_aug, train_loader_aug, test_loader, epochs=30)

# Összehasonlítás
plt.figure(figsize=(10, 5))
plt.plot(history_modern['test_acc'], label='Augmentáció nélkül')
plt.plot(history_aug['test_acc'], label='Augmentációval')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('Augmentáció hatása')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Transfer learning

### Stratégiák

| Stratégia | Leírás | Mikor |
|-----------|--------|-------|
| Feature extraction | Pretrained CNN fix, új classifier | Kevés adat |
| Fine-tuning | Pretrained CNN részleges tanítás | Közepes adat |
| Full training | Minden réteg tanítása | Sok adat |

In [ ]:
# Transfer learning szimuláció
# (Valós esetben: torchvision.models.resnet18(pretrained=True))

class PretrainedBackbone(nn.Module):
    """Szimulált pretrained backbone."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        return self.features(x).flatten(1)

class TransferModel(nn.Module):
    def __init__(self, backbone, num_classes=4, freeze_backbone=True):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(256, num_classes)

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# Feature extraction
print("Feature Extraction (frozen backbone):")
backbone = PretrainedBackbone()
model_frozen = TransferModel(backbone, freeze_backbone=True)

trainable = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_frozen.parameters())
print(f"  Tanítható: {trainable:,} / {total:,} ({trainable/total:.1%})")

# Fine-tuning
print("\nFine-tuning (unfrozen backbone):")
backbone2 = PretrainedBackbone()
model_unfrozen = TransferModel(backbone2, freeze_backbone=False)

trainable = sum(p.numel() for p in model_unfrozen.parameters() if p.requires_grad)
print(f"  Tanítható: {trainable:,} / {total:,} ({trainable/total:.1%})")

## 5. Teljes pipeline

### Konfúziós mátrix és metrikák

In [ ]:
# Evaluation

def evaluate_model(model, test_loader, class_names):
    """Modell kiértékelése."""
    device = next(model.parameters()).device
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Confusion matrix
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for true, pred in zip(all_labels, all_preds):
        cm[true, pred] += 1

    # Per-class metrics
    precision = np.diag(cm) / cm.sum(axis=0).clip(min=1)
    recall = np.diag(cm) / cm.sum(axis=1).clip(min=1)
    f1 = 2 * precision * recall / (precision + recall).clip(min=1e-8)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion matrix
    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_xticks(range(n_classes))
    axes[0].set_yticks(range(n_classes))
    axes[0].set_xticklabels(class_names, rotation=45)
    axes[0].set_yticklabels(class_names)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].set_title('Confusion Matrix')

    for i in range(n_classes):
        for j in range(n_classes):
            axes[0].text(j, i, cm[i, j], ha='center', va='center')

    plt.colorbar(im, ax=axes[0])

    # Per-class metrics
    x = np.arange(n_classes)
    width = 0.25

    axes[1].bar(x - width, precision, width, label='Precision')
    axes[1].bar(x, recall, width, label='Recall')
    axes[1].bar(x + width, f1, width, label='F1')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(class_names, rotation=45)
    axes[1].set_ylabel('Score')
    axes[1].set_title('Per-class Metrics')
    axes[1].legend()
    axes[1].set_ylim(0, 1.1)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    # Overall accuracy
    accuracy = (all_preds == all_labels).mean()
    print(f"\nOverall Accuracy: {accuracy:.1%}")
    print(f"Macro F1: {f1.mean():.3f}")

# Evaluate best model
evaluate_model(modern_cnn_aug, test_loader, class_names)

In [ ]:
# Predikciók vizualizálása

def visualize_predictions(model, X, y, class_names, n_samples=12):
    """Random predikciók megjelenítése."""
    device = next(model.parameters()).device
    model.eval()

    idx = np.random.choice(len(X), n_samples, replace=False)

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.ravel()

    for ax, i in zip(axes, idx):
        img = X[i]
        true_label = y[i]

        # Prediction
        with torch.no_grad():
            x_tensor = torch.FloatTensor(img).unsqueeze(0).to(device)
            output = model(x_tensor)
            probs = F.softmax(output, dim=1)[0]
            pred_label = probs.argmax().item()
            confidence = probs[pred_label].item()

        # Display
        img_display = img.transpose(1, 2, 0)
        img_display = np.clip(img_display, 0, 1)
        ax.imshow(img_display)

        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'Pred: {class_names[pred_label]}\n'
                    f'True: {class_names[true_label]}\n'
                    f'Conf: {confidence:.1%}', color=color)
        ax.axis('off')

    plt.suptitle('Predikciók (zöld=helyes, piros=hibás)', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_predictions(modern_cnn_aug, X_test, y_test, class_names)

## Összefoglalás

### Képklasszifikáció pipeline

| Lépés | Komponens |
|-------|----------|
| 1. Adat | DataLoader + Augmentation |
| 2. Modell | CNN (Conv + Pool + FC) |
| 3. Tanítás | CrossEntropy + Adam |
| 4. Kiértékelés | Accuracy, F1, Confusion |

### Best practices

1. **Data augmentation** - mindig használd
2. **BatchNorm** - stabilabb tanulás
3. **Global Average Pooling** - kevesebb paraméter
4. **Transfer learning** - pretrained modellek

### PyTorch kód

```python
# Modern CNN
nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),
    ...
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(channels, num_classes)
)
```